In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
import pandas as pd

df = pd.read_csv("Dataset/netflix_titles.csv")

# Netflix Content Patterns 

## Week 3: Cleaning Sprint

### Missing Value Handling

In this notebook section, missing values in the Netflix dataset are identified and analyzed. 
Appropriate cleaning strategies are selected based on the nature of each column to preserve as much useful information as possible.

## Step 1: Identifying Missing Values

Before performing any cleaning, it is important to identify the number of missing values in each column. This helps us understand the extent of incomplete data and decide whether the missing values should be replaced, retained, or handled differently.

In [4]:
df.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

## Step 2: Missing Value Summary

A summary table is created to show the number and percentage of missing values in each column. The percentage helps determine whether a column has a small or significant amount of missing data.

In [5]:
missing_values = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Percentage': (df.isnull().sum() / len(df)) * 100
})

missing_values[missing_values['Missing Values'] > 0]

,Missing Values,Percentage
director,2634,29.908028
cast,825,9.367549
country,831,9.435676
date_added,10,0.113546
rating,4,0.045418
duration,3,0.034064


## Step 3: Understanding the Missing Values

Missing values should not be removed immediately because they may contain meaningful information. For example, a missing director or cast may indicate that the information was unavailable in the original Netflix record rather than an error in the dataset.

Therefore, each column is examined separately and an appropriate treatment method is selected.

In [7]:
missing_before = df.isnull().sum()

### Handling Missing Values in the Director Column

The `director` column contains categorical and textual information. Replacing missing values with the mode could incorrectly assign a director to a movie or TV show.

Therefore, missing values are replaced with `Unknown` to indicate that the director information is not available.

In [8]:
df['director'] = df['director'].fillna('Unknown')

### Handling Missing Values in the Cast Column

The `cast` column contains the names of actors. Since assigning the most common cast to records with missing values would create incorrect information, missing values are replaced with `Unknown`.

This preserves the record while clearly indicating that cast information is unavailable.

In [9]:
df['cast'] = df['cast'].fillna('Unknown')

### Handling Missing Values in the Country Column

The `country` column contains categorical information about the production country. Replacing missing values with a specific country could introduce incorrect assumptions.

Therefore, missing values are represented as `Unknown` to preserve the original records without creating false information.

In [10]:
df['country'] = df['country'].fillna('Unknown')

### Handling Missing Values in the Date Added Column

The `date_added` column represents the date on which the content was added to Netflix. Since the actual date cannot be accurately inferred from other values, replacing it with an arbitrary date could introduce misleading information.

Therefore, missing values are retained as missing rather than assigning an incorrect date.

In [11]:
# No imputation is performed for date_added
df['date_added'].isnull().sum()

np.int64(10)

### Handling Missing Values in the Rating Column

The `rating` column contains categorical content ratings such as TV-MA, TV-14, PG, and R. Since the rating is an important attribute, missing values are replaced with the most frequently occurring rating.

The mode is selected because it represents the most common category in the dataset.

In [12]:
mode_rating = df['rating'].mode()[0]

df['rating'] = df['rating'].fillna(mode_rating)

In [13]:
mode_rating

'TV-MA'

### Handling Missing Values in the Duration Column

The `duration` column is checked before applying any treatment. If missing values are present, they are handled carefully because duration depends on the type of content, such as movies or TV shows.

No unnecessary modification is performed if the column does not contain missing values.

In [14]:
df['duration'].isnull().sum()

np.int64(3)

## Step 4: Verifying the Cleaning Process

After applying the selected missing-value treatments, the dataset is checked again to verify the results. This ensures that the cleaning process was applied correctly.

In [15]:
missing_after = df.isnull().sum()

missing_after

show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           0
duration         3
listed_in        0
description      0
dtype: int64

In [16]:
comparison = pd.DataFrame({
    'Before Cleaning': missing_before,
    'After Cleaning': missing_after
})

comparison

,Before Cleaning,After Cleaning
show_id,0,0
type,0,0
title,0,0
director,2634,0
cast,825,0
country,831,0
date_added,10,10
release_year,0,0
rating,4,0
duration,3,3


## Step 5: Final Verification

The final missing-value check is performed to confirm the current state of the dataset. Missing values that could be safely represented as `Unknown` or replaced using the mode have been treated, while meaningful missing information such as unavailable dates has been retained to avoid introducing incorrect data.

In [17]:
df.isnull().sum()

show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           0
duration         3
listed_in        0
description      0
dtype: int64

## Data Quality Report

The Data Quality Report is used to assess the overall health of the dataset before analysis.

It summarizes the dataset size, data types, missing values, duplicate records, and numerical variables.

In [ ]:
quality_report = pd.DataFrame({
    'Data Type': df.dtypes,
    'Missing Values': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2),
    'Unique Values': df.nunique()
})

quality_report

### Dataset Summary

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Duplicate Rows:", df.duplicated().sum())

## Outlier Detection using IQR

Outlier detection is performed to identify unusually high or low numerical values.

The Interquartile Range (IQR) method is used because it is simple and less affected by extreme values.

IQR = Q3 - Q1

Values below Q1 - 1.5 × IQR or above Q3 + 1.5 × IQR are identified as potential outliers.

In [ ]:
# Release year outliers

Q1 = df['release_year'].quantile(0.25)
Q3 = df['release_year'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['release_year'] < lower) | (df['release_year'] > upper)]

print("Number of outliers:", len(outliers))
print("Lower limit:", lower)
print("Upper limit:", upper)

In [ ]:
movies = df[df['type'] == 'Movie'].copy()

movies['duration_min'] = pd.to_numeric(
    movies['duration'].str.extract(r'(\d+)')[0],
    errors='coerce'
)

In [ ]:
Q1 = movies['duration_min'].quantile(0.25)
Q3 = movies['duration_min'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

duration_outliers = movies[
    (movies['duration_min'] < lower) |
    (movies['duration_min'] > upper)
]

print("Movie duration outliers:", len(duration_outliers))
print("Lower limit:", lower)
print("Upper limit:", upper)

## Outlier Treatment Decision

The detected outliers were reviewed before deciding whether to remove them.

Potential outliers were not automatically deleted because extreme values may represent valid Netflix content. The original observations are therefore retained unless there is evidence that they are incorrect.

## Cleaned Dataset

The cleaned dataset is displayed to verify that the missing-value handling process has been successfully applied.

In [18]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


## Saving the Cleaned Dataset

The cleaned Netflix dataset is saved as a new CSV file so that it can be used for further analysis in the upcoming stages of the project.

In [19]:
df.to_csv('netflix_cleaned.csv', index=False)